# DAS-ML
A workflow for DAS acoustic event identification.

### Import some necessary modules and basical functions

In [ ]:
import os
import numpy as np
import obspy
from obspy import read
import cv2
import pickle
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
import math
from matplotlib import mlab
from PIL import Image
import numpy as np
from PIL import Image, ImageEnhance


In [ ]:
"""Basic functions for data processing and visualization."""
def _nearest_pow_2(x):
    """
    Find power of two nearest to x

    >>> _nearest_pow_2(3)
    2.0
    >>> _nearest_pow_2(15)
    16.0

    :type x: float
    :param x: Number
    :rtype: int
    :return: Nearest power of 2 to x
    """
    a = math.pow(2, math.ceil(np.log2(x)))
    b = math.pow(2, math.floor(np.log2(x)))
    if abs(a - x) < abs(b - x):
        return a
    else:
        return b


def DASTrace_spatialfrequency(data,samp_rate = 125, wlen = None, mult = 8.0, per_lap=0.9, Axis=None):
    """get spectrogram in one trace"""
    samp_rate = float(samp_rate)
    # set wlen from samp_rate if not specified otherwise
    wlen=None
    if not wlen:
        wlen = 256 / samp_rate

    npts = len(data)

        # nfft needs to be an integer, otherwise a deprecation will be raised
        # XXX add condition for too many windows => calculation takes for ever
    nfft = int(_nearest_pow_2(wlen * samp_rate))

    if npts < nfft:
        msg = (f'Input signal too short ({npts} samples, window length '
                f'{wlen} seconds, nfft {nfft} samples, sampling rate '
                f'{samp_rate} Hz)')
        raise ValueError(msg)

    if mult is not None:
        mult = int(_nearest_pow_2(mult))
        mult = mult * nfft
    nlap = int(nfft * float(per_lap))
    data = data - data.mean()
    end = npts / samp_rate

        # Here we call not plt.specgram as this already produces a plot
        # matplotlib.mlab.specgram should be faster as it computes only the
        # arrays
        # XXX mlab.specgram uses fft, would be better and faster use rfft

    specgram, freq, time = mlab.specgram(data, Fs=samp_rate, NFFT=nfft,
                                            pad_to=mult, noverlap=nlap)
    if Axis == True:
        return specgram, freq, time
    elif Axis == None:
        return specgram
    

def inverted_jet_color(value):
    multiplier=1
    vmin = 0.7
    vmax = 2.8
    value = max(0, min(1, (value - vmin) / (vmax - vmin)))  #
    #value = max(0, min(1, (value - 1) / (4.3 - 1)))
    blue = int(min(255, 255 * min(1, max(0, 4 * (0.75 - value)))) * multiplier)  
    green = int(min(255, 255 * min(1, max(0, 4 * (0.5 - abs(value - 0.5))))) * multiplier)
    red = int(min(255, 255 * min(1, max(0, 4 * (value - 0.25)))) * multiplier)  
    return red, green, blue

def data2manulabel(data, fname, colortype = 'inverted_jet_color'):
    data = np.log(abs(data))
    height, width = data.shape
    image = Image.new("RGB", (width, height))
    if colortype == 'inverted_jet_color':
        for y in range(height):
            for x in range(width):
                value = data[y, x]
                color = inverted_jet_color(value)
                image.putpixel((x, y), color)
    elif colortype == 'inference_color':
        for y in range(height):
            for x in range(width):
                value = data[y, x]
                color = inference_color(value)
                image.putpixel((x, y), color)
    # visualize the image
    factor = 1  # Enhancement factor (1.0 means no enhancement)
    enhancer = ImageEnhance.Contrast(image)
    enhanced_image = enhancer.enhance(factor)
    enhanced_image = enhanced_image.transpose(Image.FLIP_TOP_BOTTOM)

    # Save the image
    new_size = (1580, 640)
    resized_image = enhanced_image.resize(new_size)
    resized_image.save(fname)
    return resized_image

### Data processing

In [ ]:
data_file = './iDAS_segment_2h/'    # data path, which the DAS data have been resampled to 100Hz
model = YOLO('./model/best.pt')    # load trained model
path_pred_img = './Silixa_YOLO_predimg'    # save path for predicted images
image_path = './Silixa_YOLO_img'    # save path for images
label_file =  'YOLO_Silixa_pred_label.pkl'    # save path for labels
if not os.path.exists(image_path):
    os.makedirs(image_path)
if not os.path.exists(path_pred_img):
    os.makedirs(path_pred_img)
    
"""Read data and generate images"""
time_step = 1000    # time step for each image
frametag = []    
datalist = os.listdir(data_file)
datalist = [f for f in datalist if f.endswith('.npy')]    # The DAS data has been preprocessed and saved as .npy files
datalist.sort()
time = 0
for pred_data in datalist:
    print(pred_data)
    total_data = np.load(os.path.join(data_file, pred_data))

    for t in range(0,total_data.shape[1],time_step):
        spatialfrequency = []
        for tr in range(total_data.shape[0]):
            spectrogram = DASTrace_spatialfrequency(total_data[tr,t:t+time_step],Axis=None)
            spatialfrequency.append(spectrogram)
        spatialfrequency = np.array(spatialfrequency)
        spatialfrequency = np.transpose(spatialfrequency,(2,1,0))

        image_name = 'Silixa' + '_{:02d}_{:04d}'.format(time,t)+'.jpg'
        save_path = os.path.join(image_path, image_name)
        dataimage = data2manulabel(np.log(spatialfrequency[0,:,:]), save_path, colortype = 'inverted_jet_color')
        print(t/100)

        results = model(dataimage,conf = 0.8)    # Predict the Acoustic events in the spectrogram

        # visualize the results
        annotated_frame = results[0].plot()
        #将带注释的帧写入视频文件
        pred_img = 'pred_' + image_name
        cv2.imwrite(os.path.join(path_pred_img, pred_img), annotated_frame)

        b = results[0].boxes.xywhn.cpu().numpy()
        a = results[0].boxes.cls.cpu().numpy()
        a = a.reshape((len(a),1))
        conf = results[0].boxes.conf.cpu().numpy()
        conf = conf.reshape((len(conf),1))
        c = np.hstack((a,b,conf))
        frametag.append(c)
        
    # Save the predicted labels
    pickle.dump(frametag, open(label_file, 'wb'))
    time += 1
    print('time %d saved'%time)


### Visualize the result in time spatial domain

In [ ]:

fpath = './YOLO_Silixa_pred_label.pkl'

with open(fpath, 'rb') as f:
    label_data = pickle.load(f)

num_iterations = len(label_data)


    # Define the colors for each label
    #colors = plt.cm.get_cmap('tab10', 10) 
colors = ['r','orange','grey','b','green']
    # Define the width for each label
line_width = 0.1


fig, ax = plt.subplots(figsize=(15, 5))
plt.subplots_adjust(top=1, bottom=0, right=1, left=0, hspace=0, wspace=0)
plt.margins(0, 0)

# visualize the results
t = 0
for d in label_data:
    for row in d:
        label = int(row[0])
        x = row[1]
        height = row[3]
        color = colors[label]
        ax.plot([t/len(label_data), t/len(label_data)], [-x-height/2, -x + height/2], color=color, linewidth=line_width)
    t += 1

ax.set_xlim(0, 1)
ax.set_ylim(-1, 0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

ax.set_xticks([])
ax.set_yticks([])
plt.savefig('Silixa_YOLO_pred.png', dpi=300)
plt.show()
